# Legacy — teste da GPU do Colab

Antes de rodar: menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.

Depois: **Ambiente de execução → Executar tudo**. Leva uns 2 minutos. No fim aparece **PASSOU** ou **NÃO PASSOU**: mande um print do resultado.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
%%bash
# Vulkan: o carregador e as ferramentas; o driver gráfico da NVIDIA já vem montado pelo Colab (libGLX_nvidia).
# a atualização pode falhar por espelhos de outros programas (ex.: o do R): não importa, segue com o que tiver
apt-get -qq update > /dev/null 2>&1 || true
apt-get -qq install -y vulkan-tools libvulkan1 > /dev/null 2>&1 || true
which vulkaninfo > /dev/null || { echo 'vulkaninfo não instalou'; }
ICD=/usr/share/vulkan/icd.d/nvidia_icd.json
if [ ! -f $ICD ]; then
  LIB=$(ls /usr/lib64-nvidia/libGLX_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0 2>/dev/null | head -1)
  mkdir -p $(dirname $ICD)
  echo '{"file_format_version":"1.0.0","ICD":{"library_path":"'$LIB'","api_version":"1.3.0"}}' > $ICD
  echo "ICD criado com $LIB"
fi
export LD_LIBRARY_PATH=/usr/lib64-nvidia:$LD_LIBRARY_PATH
vulkaninfo --summary 2>&1 | grep -E "deviceName|apiVersion|driverName" | head -6
echo ---
EXT=$(vulkaninfo 2>/dev/null | grep -oE "VK_KHR_(ray_query|acceleration_structure|ray_tracing_pipeline)" | sort -u)
echo "$EXT"
if echo "$EXT" | grep -q ray_query && echo "$EXT" | grep -q acceleration_structure && vulkaninfo --summary 2>/dev/null | grep -q -i "NVIDIA"; then
  echo; echo "========== PASSOU: a GPU do Colab roda o Legacy Engine com ray tracing =========="
else
  echo; echo "========== NÃO PASSOU: sem Vulkan com ray tracing nesta GPU =========="
fi